# ReAct AI Agent Using LangGraph

## What are we doing in this project?

In this project, we are building a small **AI Agent** that can solve a question step-by-step.

The agent does not simply give an answer immediately.

Instead, it:

1. Understands the question.
2. Thinks about what it needs to do.
3. Decides whether it needs to use a tool.
4. Uses the tool when required.
5. Gets the result from the tool.
6. Thinks again using that result.
7. Gives the final answer.

The main technologies we are using are:

| Technology | Why are we using it? |
|---|---|
| **Python** | To write the complete program |
| **Groq API** | To communicate with the AI/LLM model |
| **LLM** | To understand the question and decide what to do |
| **Calculator Tool** | To perform calculations |
| **ReAct** | To make the AI follow Reason → Act → Observe |
| **LangGraph** | To control the complete agent workflow |
| **AgentState** | To store the agent's messages and steps |

# Frameworks Used in This Project

In this project, we are using **two different frameworks to build AI Agents**:

## LangGraph

**LangGraph** is the first framework used in this project.

We use LangGraph to build a **ReAct AI Agent** and control the flow of the agent.

### What LangGraph does
# CrewAI 

In this part of the project, we use the **CrewAI framework** to build a ReAct-based AI Agent.
**CrewAI** is an AI Agent framework.

It helps us create AI agents that can:

- Understand a task
- Use an LLM
- Use tools
- Perform actions
- Complete a given task

In our project, we create a **Math Assistant Agent** using CrewAI.

---


In [ ]:
# Install Groq Python library
!pip install -q groq

In [ ]:
# Import required libraries
import os
from getpass import getpass
os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")   #  Ask for the API key securely


Enter your Groq API key: ··········


In [ ]:
# Import Groq client
from groq import Groq
MODEL_NAME="openai/gpt-oss-120b"   # # Model that we will use
# The client automatically reads OPENAI_API_KEY and OPENAI_BASE_URL from the environment
client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

# Send a minimal test message
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "user", "content": "Say hello in one short sentence."}
    ],
)

# Pull the reply text out of the response object
print("Model replied:", response.choices[0].message.content)

Model replied: Hello!


In [ ]:
# This calculator will be used as the tool by our ReAct agent
def calculator(expression: str) -> str:
    """Parse a string like 'multiply 240 0.15' or 'divide 36 3' and return the result."""
    # Split the input into three pieces: operation, first number, second number
    parts = expression.strip().split()
    if len(parts) != 3:
        return "Error: expected format 'multiply X Y' or 'divide X Y'"

    op, a_str, b_str = parts[0].lower(), parts[1], parts[2]

    # Convert the two numbers from text to float
    try:
        a = float(a_str)
        b = float(b_str)
    except ValueError:
        return "Error: numbers could not be parsed"

    # Perform the requested operation
    if op == "multiply":
        return str(a * b)
    elif op == "divide":
        if b == 0:
            return "Error: division by zero"
        return str(a / b)
    else:
        return "Error: unknown operation, use 'multiply' or 'divide'"


In [ ]:
# Instructions given to the LLM
# These instructions tell the model how to behave as a ReAct agent
SYSTEM_PROMPT = """You are a ReAct agent that solves math questions step by step.

You have exactly one tool:
  calculator(expression) - expression must be 'multiply X Y' or 'divide X Y'

On every turn you MUST reply in one of these two formats and nothing else:

Format A (when you need the tool):
THOUGHT: <one short sentence explaining your reasoning>
ACTION: <tool input, for example: multiply 240 0.15>

Format B (when you are done):
THOUGHT: <one short sentence explaining your reasoning>
FINAL ANSWER: <the final numeric answer>

Rules:
- Use ACTION to call the tool. Do not compute results in your head.
- Use only 'multiply' or 'divide'. For a percentage like 15 percent, multiply by 0.15.
- When the user question requires several steps, do them one at a time.
- Stop as soon as you can give FINAL ANSWER."""

print("System prompt prepared.")


System prompt prepared.


In [ ]:
# TypedDict is used to define the structure of our LangGraph state
from typing import TypedDict, List, Dict
class AgentState(TypedDict):
    messages: List[Dict[str, str]]      # Stores the complete conversation history
    steps: int      # Counts how many reasoning/action steps have happened

In [ ]:
# REASON NODE
# This node asks the LLM what should happen next
def reason_node(state: AgentState) -> dict:
    """Ask the LLM what to do next, based on the conversation so far."""
    print(f"\n--- Step {state['steps'] + 1}: Reasoning ---")

    # Send the full message history to the model
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=state["messages"],
    )
    reply = response.choices[0].message.content
    print("Model said:")
    print(reply)

    # Append the model's reply as an assistant message
    new_messages = state["messages"] + [{"role": "assistant", "content": reply}]
    return {"messages": new_messages, "steps": state["steps"] + 1}

In [ ]:
# This node looks for ACTION in the model's response
# and executes the calculator tool
def act_node(state: AgentState) -> dict:
    last_message = state["messages"][-1]["content"]

    if "ACTION:" in last_message:
        action = last_message.split("ACTION:")[1].strip()

        result = calculator(action)

        new_messages = state["messages"] + [
            {"role": "tool", "content": f"OBSERVATION: {result}"}
        ]

        return {
            "messages": new_messages,
            "steps": state["steps"] + 1
        }

    return state

In [ ]:
# Maximum number of graph steps
# This prevents the agent from running forever
MAX_STEPS = 4

def route(state: AgentState) -> str:
    """Decide what to do after a reasoning step."""
    # Safety limit: do not loop forever
    if state["steps"] >= MAX_STEPS:
        print(f"\n[Hit step limit of {MAX_STEPS}, stopping]")
        return "end"

    last = state["messages"][-1]["content"].upper()
    if "FINAL ANSWER:" in last:
        return "end"
    if "ACTION:" in last:
        return "act"

    # If the model did neither, stop so we do not spin
    print("\n[No ACTION or FINAL ANSWER found, stopping]")
    return "end"

In [ ]:
# Install LangGraph
!pip install -q langgraph

In [ ]:
# Import LangGraph components
from langgraph.graph import StateGraph, START, END
# Create a graph using our AgentState
builder = StateGraph(AgentState)

# Register our two nodes
builder.add_node("reason", reason_node)
builder.add_node("act", act_node)

# Entry point: always start by reasoning
builder.add_edge(START, "reason")

# After reasoning, route based on what the model produced
builder.add_conditional_edges(
    "reason",
    route,
    {"act": "act", "end": END},
)

# After acting, always go back to reasoning
builder.add_edge("act", "reason")

graph = builder.compile()
print("Graph compiled: START -> reason -> (act -> reason)* -> END")


Graph compiled: START -> reason -> (act -> reason)* -> END


In [ ]:
### 2.8 Run the Agent
# Question for our LangGraph ReAct agent

QUESTION = "What is 15 percent of 240, and then what is that result divided by 3?"

initial_state = {
    "messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": QUESTION},
    ],
    "steps": 0,
}

print("=" * 60)
print("Running LangGraph ReAct agent")
print("Question:", QUESTION)
print("=" * 60)

final_state = graph.invoke(initial_state, config={"recursion_limit": 20})  # recursion_limit is a LangGraph safety net

Running LangGraph ReAct agent
Question: What is 15 percent of 240, and then what is that result divided by 3?

--- Step 1: Reasoning ---
Model said:
THOUGHT: Dividing 36 by 3 gives 12.  
FINAL ANSWER: 12


In [ ]:
# Display everything that happened inside the graph
print("=" * 60)
print("FULL MESSAGE HISTORY")
print("=" * 60)
for i, msg in enumerate(final_state["messages"]):
    print(f"\n[{i}] {msg['role'].upper()}")
    print(msg["content"])

print("\n" + "=" * 60)
print(f"Total reasoning steps used: {final_state['steps']} (limit was {MAX_STEPS})")

FULL MESSAGE HISTORY

[0] SYSTEM
You are a ReAct agent that solves math questions step by step.

You have exactly one tool:
  calculator(expression) - expression must be 'multiply X Y' or 'divide X Y'

On every turn you MUST reply in one of these two formats and nothing else:

Format A (when you need the tool):
THOUGHT: <one short sentence explaining your reasoning>
ACTION: <tool input, for example: multiply 240 0.15>

Format B (when you are done):
THOUGHT: <one short sentence explaining your reasoning>
FINAL ANSWER: <the final numeric answer>

Rules:
- Use ACTION to call the tool. Do not compute results in your head.
- Use only 'multiply' or 'divide'. For a percentage like 15 percent, multiply by 0.15.
- When the user question requires several steps, do them one at a time.
- Stop as soon as you can give FINAL ANSWER.

[1] USER
What is 15 percent of 240, and then what is that result divided by 3?

[2] ASSISTANT
THOUGHT: Dividing 36 by 3 gives 12.  
FINAL ANSWER: 12

Total reasoning ste

# CREWAI ReAct AI AGENT

## What are we doing?

In this part, we are creating a **ReAct AI Agent using CrewAI**.

Our AI Agent will solve a **multi-step math problem**.

The important thing is that the Agent will **not do the calculation directly**.

Instead, it will use a **Calculator Tool** whenever it needs to calculate something.

### Our example question:

**What is 15% of 240, and then what is that result divided by 3?**

The Agent will solve it step by step:

    Step 1 → 15% of 240 = 36
    Step 2 → 36 ÷ 3 = 12

So,

**Final Answer = 12**


---


### Simple example:

Imagine Rahul is solving a math problem.

Rahul has a **calculator**.

He does not calculate everything in his head.

Instead:

    Question
       ↓
    Think
       ↓
    Use Calculator
       ↓
    Get Result
       ↓
    Think Again
       ↓
    Final Answer

CrewAI helps us build an AI Agent that works in a similar way.


---

# What is the aim of our project?

The main aim of this project is:

**To create a ReAct AI Agent using CrewAI that can solve a multi-step math problem by using a Calculator Tool step by step.**

Our Agent should:

1. Understand the question
2. Break the question into steps
3. Decide what to do first
4. Use the Calculator Tool
5. Get the result
6. Decide what to do next
7. Use the Calculator Tool again
8. Give the final answer


---

#What is ReAct?

**ReAct = Reason + Act**

ReAct is an approach where an AI Agent first **reasons about what it needs to do** and then **takes an action**.

The Agent can use a tool, receive the result, think again, and then continue.

### ReAct flow:

    Question
       ↓
    Reason
       ↓
    Action
       ↓
    Use Tool
       ↓
    Get Result
       ↓
    Reason Again
       ↓
    Next Action
       ↓
    Final Answer


---

# What problem are we solving?

We are solving:

**What is 15% of 240, and then what is that result divided by 3?**

The Agent should solve this in two steps.

### Step 1:

15% = 0.15

The Agent uses the Calculator Tool:

    multiply 240 0.15

Calculator returns:

    36

### Step 2:

Now the Agent uses the first result:

    divide 36 3

Calculator returns:

    12

Therefore:

**Final Answer = 12**


---

# Technologies used in our project

### 1. Python

Python is the programming language we use to build the project.

### 2. CrewAI

CrewAI is used to create and manage our AI Agent.

### 3. Groq

Groq provides the **Large Language Model (LLM)** that our Agent uses for reasoning.

### 4. Calculator Tool

The Calculator Tool performs the actual mathematical calculations.

### 5. Pydantic

Pydantic is used to define and validate the input given to our Calculator Tool.


---

# Main components of our CrewAI project

Our project mainly contains:

    CrewAI Project
          |
          ├── Agent
          |
          ├── Task
          |
          ├── Tool
          |
          └── Crew


#  Agent vs Task vs Tool vs Crew

| Component | Simple Meaning |
|---|---|
| Agent | AI worker |
| Task | Work given to the Agent |
| Tool | External capability used by the Agent |
| Calculator Tool | Performs mathematical calculations |
| Crew | Organizes Agents and Tasks |
| LLM | AI model used for reasoning |
| Groq | Provides the LLM service |
| ReAct | Reason + Act approach |

---




In [ ]:
!pip install -U "crewai[litellm]" groq  # Install CrewAI with LiteLLM support and Groq

In [ ]:
# Disable CrewAI cache_breakpoint for Groq
import crewai.llms.cache as _crewai_cache

# Keep every message unchanged
_crewai_cache.mark_cache_breakpoint = lambda msg: msg

In [ ]:
from crewai import LLM   # Import CrewAI's LLM class

MODEL_NAME = "openai/gpt-oss-120b"   # Groq model used by our CrewAI Agent

crew_llm = LLM(
    model=f"groq/{MODEL_NAME}", # Tell CrewAI to use the model through Groq
    api_key=os.environ["GROQ_API_KEY"],   # Give CrewAI the Groq API key
)

print("CrewAI LLM configured.")

CrewAI LLM configured.


In [ ]:
# Base class for creating a CrewAI tool
from crewai.tools import BaseTool
from pydantic import BaseModel, Field  # Used to define the tool input

# Input schema: tells CrewAI what arguments the tool expects
class CalculatorInput(BaseModel):
    expression: str = Field(
        ...,
        description="A string like 'multiply 240 0.15' or 'divide 36 3'",
    )

class CalculatorTool(BaseTool):
    name: str = "calculator"
    description: str = (
        "Performs one multiplication or division. "
        "Input must be a string in the form 'multiply X Y' or 'divide X Y'. "
        "For a percentage like 15 percent, multiply by 0.15."
    )
    args_schema: type = CalculatorInput

    def _run(self, expression: str) -> str:
        # Reuse the same calculator function from Part 2
        return calculator(expression)

calc_tool = CalculatorTool()
print("Tool ready:", calc_tool.name)

Tool ready: calculator


In [ ]:
# Import CrewAI Agent
from crewai import Agent

math_agent = Agent(
    role="Math Assistant",   # Role of our AI Agent
    goal="Answer math questions by calling the calculator tool step by step",  # Main goal
    backstory=(
        "You are a careful math assistant. You never compute values in your head. "
        "Whenever a calculation is needed you call the calculator tool. "
        "For a percentage like 15 percent, you multiply by 0.15."
    ),
    tools=[calc_tool],  # Give the Agent access to the calculator
    llm=crew_llm,    # Give the Agent our Groq LLM
    verbose=True,   # Show the Agent's working process
    allow_delegation=False,    # This Agent does not need another Agent
    cache=False,  # Avoid using cached results
)

print("Agent ready:", math_agent.role)

Agent ready: Math Assistant


In [ ]:
# Import Task, Crew and execution process
from crewai import Task, Crew, Process

QUESTION = "What is 15 percent of 240, and then what is that result divided by 3?"

math_task = Task(
    description=(
        f"Answer the following question using the calculator tool. "
        f"Do the calculation in two steps: first the percentage, then the division. "
        f"Never compute the numbers yourself.\n\n"
        f"Question: {QUESTION}"
    ),
    expected_output="A short sentence stating the final numeric answer.",
    agent=math_agent,
)

crew = Crew(
    agents=[math_agent],
    tasks=[math_task],
    process=Process.sequential,
    verbose=True,
)

print("Crew ready.")


Crew ready.


In [ ]:
# Run the CrewAI Agent
print("=" * 60)
print("Running CrewAI ReAct agent")
print("Question:", QUESTION)   # Display the question
print("=" * 60)

result = await crew.kickoff_async()   # Start the CrewAI task asynchronously
print("=" * 60)
print("FINAL ANSWER (CrewAI)")
print("=" * 60)
print(result)   # Print the Agent's final response

Running CrewAI ReAct agent
Question: What is 15 percent of 240, and then what is that result divided by 3?


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 98b101bf-6e3b-4d0a-a25e-cc03f8a7d5ae                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Answer the following question using the calculator tool. Do the calculation in two steps: first the      │
│  percentage, then the division. Never compute the numbers yourself.                                             │
│                                                                                                                 │
│  Question: What is 15 percent of 240, and then what is that result divided by 3?                                │
│  ID: 6db98c9c-ca9c-4aa8-9e87-6f1305a27f80                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Math Assistant                                                                                          │
│                                                                                                                 │
│  Task: Answer the following question using the calculator tool. Do the calculation in two steps: first the      │
│  percentage, then the division. Never compute the numbers yourself.                                             │
│                                                                                                                 │
│  Question: What is 15 percent of 240, and then what is that result divided by 3?                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool calculator executed with result: 36.0...


...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Math Assistant                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  15 percent of 240 is 36, and dividing that by 3 gives 12.                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Answer the following question using the calculator tool. Do the calculation in two steps: first the      │
│  percentage, then the division. Never compute the numbers yourself.                                             │
│                                                                                                                 │
│  Question: What is 15 percent of 240, and then what is that result divided by 3?                                │
│  Agent: Math Assistant                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 98b101bf-6e3b-4d0a-a25e-cc03f8a7d5ae                                                                       │
│  Final Output: 15 percent of 240 is 36, and dividing that by 3 gives 12.                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

FINAL ANSWER (CrewAI)
15 percent of 240 is 36, and dividing that by 3 gives 12.
